# 05 - Conditional Routing and Parallel Routing

How to branch a graph based on state (conditional edges), how to fan work out
to multiple nodes at once and merge results back (parallel edges + reducer),
and how to combine both (conditional fan-out to a subset of branches).
Each graph is rendered with `draw_mermaid_png` so you can see the shape.

In [ ]:
from typing import TypedDict, Annotated
import operator
from IPython.display import display, Image
from langgraph.graph import StateGraph, START, END

## Conditional Routing

A router function looks at state and returns the name of the next node.
`add_conditional_edges` wires that router in after a node.

In [ ]:
State = TypedDict("State", {"number": int, "result": str})

def check_number(state):
    return state

def even_node(state):
    return {"result": f"{state['number']} is even"}

def odd_node(state):
    return {"result": f"{state['number']} is odd"}

def route(state):
    return "even_node" if state["number"] % 2 == 0 else "odd_node"

builder = StateGraph(State)
builder.add_node("check_number", check_number)
builder.add_node("even_node", even_node)
builder.add_node("odd_node", odd_node)
builder.add_edge(START, "check_number")
builder.add_conditional_edges("check_number", route, {"even_node": "even_node", "odd_node": "odd_node"})
builder.add_edge("even_node", END)
builder.add_edge("odd_node", END)
app = builder.compile()

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
app.invoke({"number": 7, "result": ""})

## Parallel Routing (fan-out / fan-in)

Multiple edges out of the same node run in parallel branches. Since each
branch writes to the same `results` field, it needs a reducer
(`Annotated[list, operator.add]`) so results append instead of overwrite.

In [ ]:
ParallelState = TypedDict("ParallelState", {"topic": str, "results": Annotated[list, operator.add]})

def start_node(state):
    return {}

def research_a(state):
    return {"results": [f"A researched {state['topic']}"]}

def research_b(state):
    return {"results": [f"B researched {state['topic']}"]}

def research_c(state):
    return {"results": [f"C researched {state['topic']}"]}

def combine(state):
    return {"results": ["Combined: " + " | ".join(state["results"])]}

builder2 = StateGraph(ParallelState)
builder2.add_node("start", start_node)
builder2.add_node("research_a", research_a)
builder2.add_node("research_b", research_b)
builder2.add_node("research_c", research_c)
builder2.add_node("combine", combine)

builder2.add_edge(START, "start")
builder2.add_edge("start", "research_a")
builder2.add_edge("start", "research_b")
builder2.add_edge("start", "research_c")
builder2.add_edge("research_a", "combine")
builder2.add_edge("research_b", "combine")
builder2.add_edge("research_c", "combine")
builder2.add_edge("combine", END)

parallel_app = builder2.compile()

display(Image(parallel_app.get_graph().draw_mermaid_png()))

In [ ]:
parallel_app.invoke({"topic": "LangGraph", "results": []})

## Conditional Parallel Routing

A router can also return a *list* of node names instead of one - that fans out
to only that subset of branches, in parallel, based on state.

In [ ]:
State3 = TypedDict("State3", {"tasks": list, "results": Annotated[list, operator.add]})

def plan(state):
    return {}

def task_x(state):
    return {"results": ["x done"]}

def task_y(state):
    return {"results": ["y done"]}

def task_z(state):
    return {"results": ["z done"]}

def route_tasks(state):
    return state["tasks"]  # e.g. ["task_x", "task_z"] -> only those run, in parallel

builder3 = StateGraph(State3)
builder3.add_node("plan", plan)
builder3.add_node("task_x", task_x)
builder3.add_node("task_y", task_y)
builder3.add_node("task_z", task_z)
builder3.add_edge(START, "plan")
builder3.add_conditional_edges("plan", route_tasks, ["task_x", "task_y", "task_z"])
builder3.add_edge("task_x", END)
builder3.add_edge("task_y", END)
builder3.add_edge("task_z", END)

dynamic_app = builder3.compile()

display(Image(dynamic_app.get_graph().draw_mermaid_png()))

In [ ]:
dynamic_app.invoke({"tasks": ["task_x", "task_z"], "results": []})